In [ ]:
# 01 Process Data
# Load CheXpert+ metadata, labels, and image paths.
# Then select one frontal image per study, so each final row is one study.
# Create a probe train/test split and save a CSV.

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)

In [ ]:
# Set paths, labels, and split sizes.

CXR_DIR = Path("/opt/gpudata/cxr")
CHEXPERTPLUS_DIR = CXR_DIR / "chexpertplus"
LABEL_DIR = CXR_DIR / "derived"

IMAGE_ROOT = CHEXPERTPLUS_DIR / "PNG"
SPLIT_CSV = CHEXPERTPLUS_DIR / "split.csv"
METADATA_CSV = CHEXPERTPLUS_DIR / "metadata.csv"
LABEL_CSV = LABEL_DIR / "chexpertplus-findings-labels-chexbert.csv"

TARGET_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]

TRAIN_STUDIES_N = 20_000
TEST_STUDIES_N = 5_000
RANDOM_SEED = 42

OUT_PATH = Path("artifacts/processed_data/chexpertplus_frontal_5labels.csv")

for path in [IMAGE_ROOT, SPLIT_CSV, METADATA_CSV, LABEL_CSV]:
    print(f"{path}: {path.exists()}")

In [ ]:
# Load the split table, metadata table, and CheXpert labels.

split_df = pd.read_csv(SPLIT_CSV)
meta_df = pd.read_csv(METADATA_CSV)
label_df = pd.read_csv(LABEL_CSV)

print("split_df", split_df.shape)
display(split_df.head())

print("meta_df", meta_df.shape)
display(meta_df.head())

print("label_df", label_df.shape)
display(label_df[["study_id"] + TARGET_LABELS].head())

In [ ]:
# Scan image files and create IDs that match the metadata tables.
# CheXpert+ image paths are expected to end with patient_id / raw_study_id / raw_dicom_id.png.
# The normalized IDs are study_id = patient_id_raw_study_id and dicom_id = study_id_raw_dicom_id.

rows = []
image_paths = sorted(IMAGE_ROOT.rglob("*.png"))

for path in tqdm(image_paths, desc="Scanning images"):
    patient_id, raw_study_id, raw_dicom_id = path.with_suffix("").parts[-3:]
    study_id = f"{patient_id}_{raw_study_id}"
    dicom_id = f"{study_id}_{raw_dicom_id}"
    rows.append(
        {
            "subject_id": patient_id,
            "study_id": study_id,
            "dicom_id": dicom_id,
            "image_path": str(path),
        }
    )

image_df = pd.DataFrame(rows)

# patient32368 has an image-loading problem.
image_df = image_df[image_df["subject_id"] != "patient32368"].reset_index(drop=True)

print("image_df", image_df.shape)
display(image_df.head())

In [ ]:
# Merge split, metadata, image paths, and labels.

meta_small = meta_df[["subject_id", "study_id", "dicom_id", "ViewPosition"]].copy()
meta_small["view"] = meta_small["ViewPosition"].fillna("").str.upper()

data = (
    split_df[["study_id", "split"]]
    .drop_duplicates("study_id")
    .merge(meta_small, on="study_id", how="inner")
    .merge(image_df, on=["subject_id", "study_id", "dicom_id"], how="left")
    .merge(label_df[["study_id"] + TARGET_LABELS], on="study_id", how="inner")
)

data["has_image"] = data["image_path"].notna()

print("data", data.shape)
display(data.head())

In [ ]:
# Print basic statistics before filtering.

print("Rows:", len(data))
print("Studies:", data["study_id"].nunique())
print("Patients:", data["subject_id"].nunique())

print("View distribution")
display(data["view"].value_counts(dropna=False).to_frame("rows"))

print("Rows with image paths:", int(data["has_image"].sum()))

In [ ]:
# Keep PA/AP images only.
# If a study has multiple frontal images, prefer PA, then AP.
# After this step, each row is one study.

frontal = data[data["view"].isin(["PA", "AP"]) & data["has_image"]].copy()
frontal["view_rank"] = frontal["view"].map({"PA": 0, "AP": 1})

manifest = (
    frontal.sort_values(["study_id", "view_rank", "dicom_id"])
    .drop_duplicates("study_id", keep="first")
    .sort_values(["split", "subject_id", "study_id"])
    .reset_index(drop=True)
)

print("manifest", manifest.shape)
display(manifest.head())

print("Rows:", len(manifest))
print("Studies:", manifest["study_id"].nunique())
print("Patients:", manifest["subject_id"].nunique())

print("View distribution after filtering")
display(manifest["view"].value_counts(dropna=False).to_frame("rows"))

print("Each row is one study:", len(manifest) == manifest["study_id"].nunique())

In [ ]:
# Create a probe split where each row is one study.
# MedGemma was not trained on CheXpert+, so we do not need to follow the original train/validate/test split.
# We split the data ourselves: 20,000 train studies and 5,000 test studies.
# We still assign split membership at the patient level, so the same patient cannot appear in both train and test.

patient_counts = (
    manifest.groupby("subject_id")
    .size()
    .rename("n_rows")
    .reset_index()
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

train_patients = []
test_patients = []
train_rows = 0
test_rows = 0

for row in patient_counts.itertuples(index=False):
    if train_rows + row.n_rows <= TRAIN_STUDIES_N:
        train_patients.append(row.subject_id)
        train_rows += row.n_rows
    elif test_rows + row.n_rows <= TEST_STUDIES_N:
        test_patients.append(row.subject_id)
        test_rows += row.n_rows
    if train_rows == TRAIN_STUDIES_N and test_rows == TEST_STUDIES_N:
        break

manifest["probe_split"] = "unused"
manifest.loc[manifest["subject_id"].isin(train_patients), "probe_split"] = "train"
manifest.loc[manifest["subject_id"].isin(test_patients), "probe_split"] = "test"

probe_data = manifest[manifest["probe_split"].isin(["train", "test"])].copy()

print("probe_data", probe_data.shape)
display(
    probe_data.groupby("probe_split")
    .agg(rows=("study_id", "size"), studies=("study_id", "nunique"), patients=("subject_id", "nunique"))
)

patient_probe_split_counts = probe_data.groupby("subject_id")["probe_split"].nunique()
print("Patients in both train and test:", int((patient_probe_split_counts > 1).sum()))

In [ ]:
# Print label statistics for the five target findings.
# The primary probing rule will be positive = label == 1, and everything else is negative.

label_rows = []

for split, split_data in probe_data.groupby("probe_split"):
    for label in TARGET_LABELS:
        s = split_data[label]
        label_rows.append(
            {
                "probe_split": split,
                "label": label,
                "n": len(s),
                "positive_1": int((s == 1).sum()),
                "negative_0": int((s == 0).sum()),
                "uncertain_-1": int((s == -1).sum()),
                "missing": int(s.isna().sum()),
                "primary_positive_rate": (s == 1).mean(),
            }
        )

label_stats = pd.DataFrame(label_rows)
display(label_stats)

In [ ]:
# Save the processed dataset.

columns_to_save = [
    "subject_id",
    "study_id",
    "dicom_id",
    "split",
    "probe_split",
    "ViewPosition",
    "view",
    "image_path",
] + TARGET_LABELS

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
probe_data[columns_to_save].to_csv(OUT_PATH, index=False)

print(f"Saved {len(probe_data):,} rows to {OUT_PATH}")